# HRAI API demo notebook

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

!git lfs pull

In [ ]:
import os, json, random, textwrap, requests
from pathlib import Path

from config import conf
from load import get_encoder
from query import query_type, extract_from_resume
from suggestions import get_expanded_skills, get_domain_reports
from pos_extraction import text_to_ngrams

from dotenv import load_dotenv
import faiss

BASE_URL = 'http://127.0.0.1:8000'

In [2]:
load_dotenv()

True

## konfigurace modelu a metadat pro lokální volání
\*pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN v .env

In [4]:
db = faiss.read_index(os.path.join(conf.db_dir, f"all.index"))
with open(os.path.join(conf.data_dir, f"key_to_ent.json"),'r') as f:
    metadata = json.loads(f.read())
model = get_encoder()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
def print_skill(s):
    print(f"{s['label']}")
    if s['description']: print(f"informace: {s['description']}\n")


def print_occupation(occ):
    print(
        f"{occ['label']}\n"
        f"skóre: {round(occ['cosine_score'],5)}"
    )
    if occ['description']: print(f"info: {occ['description']}\n")


def print_missing_skills(skills):
    for s in skills:
        if 'missing_skills' in s and len(s['missing_skills']) == 0: continue
        if 'occupation' in s: print_occupation(s['occupation'])
        if 'missing_skills' not in s: continue
        essential = [s for s in s['missing_skills']if s['relation_type'] == 'essential']
        optional = [s for s in s['missing_skills']if s['relation_type'] == 'optional']

        print('Základní dovednosti/znalosti pro výkon pozice:')
        for e in essential: print_skill(e)

        print('\nDalší užitečné dovednosti:')
        for o in optional: print_skill(o)
        print('-'*100)


def print_domain_info(d):
    for occ in d['occupations']:
        if not 'missing_skills' in occ: continue
        print_occupation(occ['occupation'])
        if 'missing_skills' in occ and len(occ['missing_skills']) > 0:
            print('')
            print_missing_skills([occ])
    print('-'*100+'\n')

## převedení textu na ngramy

In [6]:
resumes = json.loads(Path(os.path.join('testfiles', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)
print(textwrap.shorten(text, width=2000))

Motivovaný profesionální stavební manažer s vynikajícími interpersonálními dovednostmi. Pracuje včas a efektivně, aby dotáhl náročné úkoly až do konce. Přinášíme cenné zkušenosti z velkých firemních stavenišť, ale i z projektů oprav rezidenčních domů. Přednosti Certifikovaná CPR a standardní první pomoc Manažer shody s dešťovou vodou Ultraweld Školení exotermického připojení Znalosti transformátorů, vysokonapěťových rozváděčů, automatického spínače tansformeru, jednofázového a třífázového napájení zběhlý v; MS Excel, MS Word Vynikající zákaznický servis, řešení konfliktů a prioritizace pracovních míst Zkušenosti Město, Státní stavební jesličky Poskytoval přesná měření a odhady pro všechny projekty a plnil rozpočtová očekávání. Horolezectví a práce na komunikačních věžích za účelem instalace, výměny a oprav zařízení anténních systémů; prováděl údržbu věže pod přísným dohledem. Jako zkušený správce věže vést výstavbu, instalaci a údržbu komunikací k věžím a podpůrným konstrukcím. Impleme

In [7]:
ngrams = text_to_ngrams(text)
# print('\n'.join(ngrams[:20])+'\n[...]')
print(f'ngramy: {len(ngrams)}\n'
      f'text: {len(text.split())}'
      )
print('\n'.join(ngrams))

ngramy: 639
text: 610
zkušený správce
dodavatelů a pracovišť
rozpočtu vývoj rozpočtu
míst po dokončení
úkoly podřízeným pracovníkům
komunikační místa
specifikací
cpr klient spokojenost
složitých problémů
transformátorů
věže vést výstavbu
přednosti
zákony nebo dodržování
zákazníků
procházek a připomínek
exotermického připojení
servis řešení konfliktů
šplhací věže
rozsah
antén
předpisů řešení
začátku do konce
úsporu energie věží
hodnotil výkon
technologií
škola
styčný bod
posádku od začátku
materiály a zařízení
desky
návrh a výstavbu
spínač
zlepšení efektivity
výstavby
například utažením šroubů
vývoj rozpočtu rozpočty
milionů dolarů
bod
jediný styčný bod
spolupracoval se správci
administrativní plán
excel ms word
orientovaný směr úspora
prováděl údržbu
harmonogramu
bázi
trhu
řízení trh
dodrženy termíny
spolupráce
rozváděč transformátory utility
celkového výkonu realizace
diplom el dorado
výměny
zadaných stavebních projektech
dodavatele a dodávky
požadavcích
manažer shody
dokončené projek

## API endpointy: /resume/skills a /resume/domains (pouze PDF)

In [8]:
def get_skills(filename: str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/pdf')}
    res = requests.post(f"{BASE_URL}/resume/skills", files=files)

    print(f'file: {filename}\n'
          f'status: {res.status_code}\n'
    )
    print_missing_skills(res.json())


def get_domains(filename:str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/pdf')}
    res = requests.post(f"{BASE_URL}/resume/domains", files=files)

    print(f'file: {filename}\n'
          f'status: {res.status_code}\n'
    )

    for d in res.json():
        print(f"oblast: {d['domain']}\n"
          f"počet zaměstnání se shodou: {d['occ_count']}\n"
          )
        if 'occupations' not in d: continue
        if len(d['occupations']) <1: continue
        print_domain_info(d)


In [10]:
get_skills('resume.pdf')

file: resume.pdf
status: 200

mluvčí
skóre: 1.0
info: Mluvčí hovoří jménem společností nebo organizací. Využívají komunikační strategie k zastupování klientů prostřednictvím veřejných oznámení a konferencí. Vykreslují své klienty v pozitivním světle a snaží se blíže vysvětlit jejich zájmy a činnost.

Základní dovednosti/znalosti pro výkon pozice:
diplomatické zásady
informace: Postupy přispívající k uzavírání dohod nebo mezinárodních smluv s jinými zeměmi prostřednictvím vedení jednání a vynakládání úsilí na ochranu zájmů domácí vlády, jakož i postupy usilující o vyjednání kompromisu.

utváření veřejného mínění
informace: Proces, v jehož rámci se formulují a prosazují názory a stanoviska ve vztahu k něčemu. Prvky, které hrají roli ve veřejném mínění, jako je formování informací, psychické procesy a stádní chování.

rétorika
informace: Umění projevu, které si klade za cíl zlepšit schopnost autorů a řečníků informovat, přesvědčovat nebo motivovat své publikum.

komunikační zásady
informa

In [18]:
get_domains('Životopis.pdf')

file: Životopis.pdf
status: 200

oblast: Řidiči a doprava
počet zaměstnání se shodou: 3

Řidiči nákladních automobilů, tahačů a speciálních vozidel
skóre: 0.73592
osobní řidič/osobní řidička / osobní řidič
skóre: 0.7261
info: Osobní řidiči přepravují své zaměstnavatele na konkrétní místo určení bezpečně a včas. Používají navigační zařízení k co nejrychlejšímu dosažení místa určení, využívají poradenství v oblasti počasí a dopravního provozu a dodržují právní předpisy v oblasti řízení.


osobní řidič/osobní řidička / osobní řidič
skóre: 0.7261
info: Osobní řidiči přepravují své zaměstnavatele na konkrétní místo určení bezpečně a včas. Používají navigační zařízení k co nejrychlejšímu dosažení místa určení, využívají poradenství v oblasti počasí a dopravního provozu a dodržují právní předpisy v oblasti řízení.

Základní dovednosti/znalosti pro výkon pozice:
geografické oblasti
informace: Detailní znalosti o geografické oblasti; znalosti o tom, kde různé organizace působí.

právní předpisy

## API endpointy: /text/skills a /text/domains

In [15]:
text_req = {
    'occupations': ['software developer', 'data analyst'],
    'skills': ['python', 'strojové učení', 'sql databáze']
}
skills_resp = requests.post(f"{BASE_URL}/text/skills", json=text_req)
print(f"endpoint: /text/skills | status: {skills_resp.status_code}\n")
print_missing_skills(skills_resp.json())


domains_resp = requests.post(f"{BASE_URL}/text/domains", json=text_req)
print(f"endpoint: /text/domains | status: {domains_resp.status_code}\n")
for d in domains_resp.json():
    print(f"oblast: {d['domain']}\n"
          f"počet zaměstnání se shodou: {d['occ_count']}\n"
          )
    if 'occupations' not in d or len(d['occupations']) < 1: continue
    print_domain_info(d)

endpoint: /text/skills | status: 200

softwarový vývojář/softwarová vývojářka / programátor softwaru
skóre: 0.86697
info: Vývojáři softwaru implementují nebo programují všechny typy softwarových systémů založených na specifikacích a návrzích s použitím programovacích jazyků, nástrojů a platforem.

Základní dovednosti/znalosti pro výkon pozice:
technické zásady
informace: Technická a strojírenská hlediska, jako je funkčnost, reprodukovatelnost a náklady související s koncepcí či návrhem, a způsob, jakým jsou uplatňovány při dokončování inženýrských projektů.

projektový management
informace: Porozumění řízení projektů a činnostem, které jsou součástí této oblasti. Znalost různých aspektů řízení projektů, jako jsou čas, zdroje, požadavky, lhůty a reakce na neočekávané události.

konstrukční procesy
informace: Systematický přístup k vývoji a údržbě inženýrských systémů.

poskytovat technickou dokumentaci
informace: Připravit dokumentaci pro stávající a budoucí výrobky nebo služby popisují

## API endpoint: /query (každý typ dotazu)

In [ ]:
from main import QueryRequest
# TODO fix example request

query = QueryRequest(
    query='manažerka v mcdonalds',
    query_type='occupation'
)

resp = requests.post(f"{BASE_URL}/query", json=query.model_dump())
print({'endpoint': '/query', 'status': resp.status_code, 'json': resp.json()})

## Přímé volání query_type pro všechny ent_type

In [14]:
entry = ['Sítě a navazování vztahů']
ent_type = 'skill'

results = query_type(db=db, metadata=metadata, model=model, ents=entry, ent_type=ent_type, min_score=0.5)
print(f"{entry} ({ent_type})")
print(results)

['Sítě a navazování vztahů'] (skill)
[EntityResult(id='23104', cosine_score=0.6837359666824341, entity_type='skill', esco_uri='http://data.europa.eu/esco/skill/bf983f2a-e941-4646-811d-cc81060cb53f', label='vytvářet sítě', code='', description='Prokazovat schopnost budovat fungující vztahy, navazovat a udržovat spojenectví, kontakty a partnerství a vyměňovat informace s ostatními.'), EntityResult(id='23961', cosine_score=0.6455310583114624, entity_type='skill', esco_uri='http://data.europa.eu/esco/skill/cc922128-f687-414a-95e0-286597e065ad', label='udržovat sítě', code='', description='Provádět výměny klecové sítě a opravy sítě na ptáky.'), EntityResult(id='24927', cosine_score=0.6448580622673035, entity_type='skill', esco_uri='http://data.europa.eu/esco/skill/dc72ad0a-c5dc-4abd-bc0d-ca43e82162e1', label='budovat obchodní vztahy', code='', description='Vytvářet pozitivní, dlouhodobý vztah mezi organizací a zúčastněnými třetími stranami, jako jsou dodavatelé, distributoři, akcionáři a da

## Pipeline: extract_from_resume -> get_expanded_skills -> get_domain_reports

In [ ]:
random_resume = random.choice(resumes)
print(textwrap.shorten(f'sample text: {random_resume}', width=2000, placeholder="..."))
print('-'*80)

extracted_entities = extract_from_resume(db, metadata, model, random_resume)
print(textwrap.shorten(f'extracted entities: {extracted_entities}', width=2000, placeholder="..."))
print('-'*80)

skill_suggestions = get_expanded_skills(metadata, extracted_entities)
print(textwrap.shorten(f'skill suggestions: {skill_suggestions}', width=2000, placeholder="..."))

domain_reports = get_domain_reports(skill_suggestions)
print(textwrap.shorten(f'domain stats: {domain_reports}', width=2000, placeholder="..."))